## Loading the data
Si hay multiwords, se debe saltar una linea y coger las dos siguientes. Ejemplo:

19-20	don't	_	_	_	_	_	_	_	_

19	do	do	AUX	VBP	Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbForm=Fin	21	aux	21:aux	_

20	n't	not	PART	RB	Polarity=Neg	21	advmod	21:advmod	_

In [5]:
def load_conllu_data(filepath):
    """
    Carga y procesa un archivo CoNLL-U, extrayendo las oraciones y sus etiquetas UPOS.
    """
    sentences = []
    tags = []
    current_sentence = []
    current_tags = []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            # 1. Ignorar comentarios y líneas vacías que no sean separadores de oración
            if line.startswith('#'):
                continue
            
            # 2. Línea en blanco: indica el final de una oración
            elif line == '':
                if current_sentence:
                    sentences.append(current_sentence)
                    tags.append(current_tags)
                    current_sentence = []
                    current_tags = []
            
            # 3. Procesar línea de palabra
            else:
                fields = line.split('\t')
                
                # Ignorar tokens multiword (ID con guion, e.g., '1-2') o nodos vacíos (ID con punto, e.g., '1.1') 
                if '-' in fields[0] or '.' in fields[0]:
                    continue

                # Extraer la palabra (FORM, índice 1) y la etiqueta PoS (UPOS, índice 3)
                word = fields[1]
                pos_tag = fields[3]
                
                current_sentence.append(word)
                current_tags.append(pos_tag)

    # Asegurarse de añadir la última oración si el archivo no termina en línea vacía
    if current_sentence:
        sentences.append(current_sentence)
        tags.append(current_tags)

    return sentences, tags

In [6]:
train_sents, train_tags = load_conllu_data('en_ewt-ud-train.conllu')
dev_sents, dev_tags = load_conllu_data('en_ewt-ud-dev.conllu')
test_sents, test_tags = load_conllu_data('en_ewt-ud-test.conllu')

## 2. Text Vectorization: Creating the Dictionaries

The first step in preparing the data for the LSTM model is to convert our text-based sentences and tags into numerical sequences. Neural networks can only process numbers, so we need a consistent way to map each word and each tag to a unique integer ID.

For this task, we'll use Keras's modern `TextVectorization` layer. We will create two separate instances of this layer: one for the input words (`word_vectorizer`) and one for the output tags (`tag_vectorizer`).

The process involves two main stages:
1.  **Configuration**: We initialize the `TextVectorization` layer with `output_mode='int'` to ensure it produces sequences of integer IDs (e.g., "Google is nice" -> `[2, 3, 42]`). We also set `output_sequence_length=128` to enforce that all sequences are padded or truncated to a fixed length, which is a requirement for the model.
2.  **Adaptation**: We then call the `.adapt()` method on our training data. This step builds the internal vocabulary for each vectorizer. It analyzes all the words (or tags) in the training set and assigns a unique integer to each one. This ensures our "dictionaries" are based only on the data the model is allowed to learn from.


In [7]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

MAX_LEN = 128

def create_and_adapt_vectorizer(sentences, standardize, max_len=MAX_LEN):

    vectorizer = TextVectorization(
        output_mode='int',
        output_sequence_length=max_len,
        standardize=standardize
    )

    sentences_flat = [' '.join(sentence) for sentence in sentences]
    
    vectorizer.adapt(sentences_flat)
    
    vocab_size = len(vectorizer.get_vocabulary())
    
    print(f"Adaptation complete. Vocabulary size: {vocab_size}")
    
    return vectorizer, vocab_size


# 2. Pasa 'standardize=None' al vectorizador de palabras
word_vectorizer, WORD_VOCAB_SIZE = create_and_adapt_vectorizer(train_sents, standardize="lower")

# 3. El vectorizador de etiquetas déjalo como estaba (no necesita el cambio)
tags_vectorizer, TAGS_VOCAB_SIZE = create_and_adapt_vectorizer(train_tags, standardize='lower_and_strip_punctuation')
# Obtenemos la lista de strings del vocabulario
tags_vocab_list = tags_vectorizer.get_vocabulary()
print("tags_vectorizer",tags_vocab_list)

print(f"\nWe have successfully created a vectorizer with a vocabulary of {WORD_VOCAB_SIZE} words.")

print(f"\nWe have successfully created a vectorizer with a vocabulary of {TAGS_VOCAB_SIZE} tags.")

print("Vectorizing all data sets...")

# --- 1. Flatten the data from list of lists to list of strings ---
# The vectorizer layers expect a flat list of strings as input.
train_sents_flat = [' '.join(sentence) for sentence in train_sents]
train_tags_flat = [' '.join(tag_list) for tag_list in train_tags]

dev_sents_flat = [' '.join(sentence) for sentence in dev_sents]
dev_tags_flat = [' '.join(tag_list) for tag_list in dev_tags]

test_sents_flat = [' '.join(sentence) for sentence in test_sents]
test_tags_flat = [' '.join(tag_list) for tag_list in test_tags]


# --- 2. Use the vectorizers to transform the flattened data ---
# Now we call the vectorizers with the correct input format.
X_train = word_vectorizer(train_sents_flat)
y_train = tags_vectorizer(train_tags_flat)

X_dev = word_vectorizer(dev_sents_flat)
y_dev = tags_vectorizer(dev_tags_flat)

X_test = word_vectorizer(test_sents_flat)
y_test = tags_vectorizer(test_tags_flat)

print("Vectorization complete!")
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_dev:", X_dev.shape)
print("Shape of y_dev:", y_dev.shape)


print("X_train",X_train[0])
print("y_train",y_train[0])
print("X_test",X_test[0])
print("y_test",y_test[0])
print(TAGS_VOCAB_SIZE)

2025-10-29 18:09:33.897800: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-29 18:09:33.899104: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-29 18:09:33.995439: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto.

Adaptation complete. Vocabulary size: 16656
Adaptation complete. Vocabulary size: 19
tags_vectorizer ['', '[UNK]', np.str_('noun'), np.str_('punct'), np.str_('verb'), np.str_('pron'), np.str_('adp'), np.str_('det'), np.str_('adj'), np.str_('aux'), np.str_('propn'), np.str_('adv'), np.str_('cconj'), np.str_('part'), np.str_('num'), np.str_('sconj'), np.str_('sym'), np.str_('intj'), np.str_('x')]

We have successfully created a vectorizer with a vocabulary of 16656 words.

We have successfully created a vectorizer with a vocabulary of 19 tags.
Vectorizing all data sets...
Vectorization complete!

Shape of X_train: (12544, 128)
Shape of y_train: (12544, 128)
Shape of X_dev: (2001, 128)
Shape of y_dev: (2001, 128)
X_train tf.Tensor(
[  185    16  4830    48   276   669  1156  4095 15915   185    16 15663
     4     2  6984    36     2  7243    10     2   460     8  6913     4
   755     2  2158  1641     3     0     0     0     0     0     0     0
     0     0     0     0     0     0     0

## Esto es el modeo

In [9]:
import tensorflow as tf
# Importamos las capas extra que necesitamos: Bidirectional y Dropout
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, TimeDistributed, Bidirectional, Dropout
from tensorflow.keras.models import Model

EMBEDDING_DIM = 64
LSTM_UNITS = 64
DROPOUT_RATE = 0.3


# 1. Definir la capa de Entrada
inputs = Input(shape=(MAX_LEN,), name='word_ids_input')

# 2. Conectar las capas
# La capa Embedding recibe los 'inputs'
x = Embedding(
    input_dim=WORD_VOCAB_SIZE, 
    output_dim=EMBEDDING_DIM, 
    mask_zero=True, 
    name='word_embedding'
)(inputs)

# Añadimos Dropout después del embedding
#x = Dropout(DROPOUT_RATE, name='dropout_embedding')(x)


# 3. Primera capa Bidirectional LSTM
x = Bidirectional(
    LSTM(
        units=LSTM_UNITS, 
        return_sequences=True,
        name='bidirectional_lstm_1'
    )
)(x)

# 5. Capa de Salida (igual que antes)
outputs = TimeDistributed(
    Dense(units=TAGS_VOCAB_SIZE, activation='softmax'), 
    name='pos_tag_output'
)(x)

# 6. Crear el Modelo final
model = Model(inputs=inputs, outputs=outputs, name='pos_tagger_model_v2_bidirectional')

# ¡Listo! Ahora puedes imprimir el resumen y ver la nueva arquitectura
model.summary()

Model: "pos_tagger_model_v2_bidirectional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ word_ids_input      │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embedding      │ (None, 128, 64)   │  1,065,984 │ word_ids_input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 128)       │          0 │ word_ids_input[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 128, 128)  │     66,048 │ word_embedding[0… │
│ (Bidirectional)     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pos_tag_output      │ (None, 128, 19)   │      2,451 │ bidirectional_1[… │
│ (TimeDistributed)   │                   │            │ not_equal_1[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,134,483 (4.33 MB)

 Trainable params: 1,134,483 (4.33 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
import tensorflow as tf
from tensorflow.keras.metrics import sparse_categorical_accuracy

# Compilar el modelo
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [12]:
# 1. Importa el Callback
from tensorflow.keras.callbacks import EarlyStopping

# 2. Crea una instancia del callback
early_stopping_callback_acc = EarlyStopping(
    monitor='val_accuracy', 
    patience=3,
    min_delta=0.001,          # Una mejora de 0.1% 
    mode='max',               # Queremos maximizar la precisión (accuracy)
    restore_best_weights=True
)

# 3. Pasa el callback a model.fit()

# Parámetros de entrenamiento
EPOCHS = 50
BATCH_SIZE = 64


# Entrenar el modelo
history = model.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_dev, y_dev),
    callbacks=[early_stopping_callback_acc]
)

Epoch 1/50


2025-10-29 18:26:10.005355: E tensorflow/core/util/util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


196/196 ━━━━━━━━━━━━━━━━━━━━ 47s 214ms/step - accuracy: 0.5612 - loss: 1.4902 - val_accuracy: 0.8285 - val_loss: 0.6236
Epoch 2/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 42s 212ms/step - accuracy: 0.9075 - loss: 0.3466 - val_accuracy: 0.9061 - val_loss: 0.3447
Epoch 3/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 43s 221ms/step - accuracy: 0.9535 - loss: 0.1762 - val_accuracy: 0.9123 - val_loss: 0.3088
Epoch 4/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 40s 203ms/step - accuracy: 0.9665 - loss: 0.1218 - val_accuracy: 0.9176 - val_loss: 0.2920
Epoch 5/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 39s 201ms/step - accuracy: 0.9733 - loss: 0.0948 - val_accuracy: 0.9168 - val_loss: 0.3015
Epoch 6/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 40s 206ms/step - accuracy: 0.9783 - loss: 0.0768 - val_accuracy: 0.9167 - val_loss: 0.3004
Epoch 7/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 40s 206ms/step - accuracy: 0.9825 - loss: 0.0632 - val_accuracy: 0.9146 - val_loss: 0.3205


Epoch 1/50


2025-10-29 18:26:10.005355: E tensorflow/core/util/util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


 16/196 ━━━━━━━━━━━━━━━━━━━━ 35s 198ms/step - accuracy: 0.1409 - loss: 2.9136

In [13]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, TimeDistributed, Bidirectional, Dropout
from tensorflow.keras.models import Model
import tensorflow as tf
from tensorflow.keras.metrics import sparse_categorical_accuracy
from tensorflow.keras.callbacks import EarlyStopping

EMBEDDING_DIM = 32
LSTM_UNITS = 64


# 1. Definir la capa de Entrada
inpusts_1 = Input(shape=(MAX_LEN,), name='word_ids_input')

# 2. Conectar las capas
# La capa Embedding recibe los 'inpusts_1'
x = Embedding(
    input_dim=WORD_VOCAB_SIZE, 
    output_dim=EMBEDDING_DIM, 
    mask_zero=True, 
    name='word_embedding'
)(inpusts_1)



# 3. Primera capa Bidirectional LSTM
x = Bidirectional(
    LSTM(
        units=LSTM_UNITS, 
        return_sequences=True,
        name='bidirectional_lstm_1'
    )
)(x)

# 5. Capa de Salida (igual que antes)
outputs = TimeDistributed(
    Dense(units=TAGS_VOCAB_SIZE, activation='softmax'), 
    name='pos_tag_output'
)(x)

# 6. Crear el Modelo final
model_1 = Model(inputs=inpusts_1, outputs=outputs, name='pos_tagger_model_v2_bidirectional')

# ¡Listo! Ahora puedes imprimir el resumen y ver la nueva arquitectura
model_1.summary()

# Compilar el modelo
model_1.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
# 2. Crea una instancia del callback
early_stopping_callback_acc = EarlyStopping(
    monitor='val_accuracy', 
    patience=3,
    min_delta=0.001,          # Una mejora de 0.1% 
    mode='max',               # Queremos maximizar la precisión (accuracy)
    restore_best_weights=True
)

# 3. Pasa el callback a model.fit()

# Parámetros de entrenamiento
EPOCHS = 50
BATCH_SIZE = 64


# Entrenar el modelo
history_1 = model_1.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_dev, y_dev),
    callbacks=[early_stopping_callback_acc]
)

Model: "pos_tagger_model_v2_bidirectional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ word_ids_input      │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embedding      │ (None, 128, 32)   │    532,992 │ word_ids_input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 128)       │          0 │ word_ids_input[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 128, 128)  │     49,664 │ word_embedding[0… │
│ (Bidirectional)     │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pos_tag_output      │ (None, 128, 19)   │      2,451 │ bidirectional_2[… │
│ (TimeDistributed)   │                   │            │ not_equal_2[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 585,107 (2.23 MB)

 Trainable params: 585,107 (2.23 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 44s 201ms/step - accuracy: 0.5038 - loss: 1.6692 - val_accuracy: 0.7761 - val_loss: 0.7667
Epoch 2/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 39s 199ms/step - accuracy: 0.8757 - loss: 0.4526 - val_accuracy: 0.8872 - val_loss: 0.3991
Epoch 3/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 23s 118ms/step - accuracy: 0.9412 - loss: 0.2265 - val_accuracy: 0.9030 - val_loss: 0.3250
Epoch 4/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 23s 115ms/step - accuracy: 0.9594 - loss: 0.1527 - val_accuracy: 0.9081 - val_loss: 0.3085
Epoch 5/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 22s 114ms/step - accuracy: 0.9675 - loss: 0.1176 - val_accuracy: 0.9083 - val_loss: 0.3042
Epoch 6/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 22s 114ms/step - accuracy: 0.9732 - loss: 0.0967 - val_accuracy: 0.9116 - val_loss: 0.3095
Epoch 7/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 23s 117ms/step - accuracy: 0.9771 - loss: 0.0815 - val_accuracy: 0.9115 - val_loss: 0.3108
Epoch 8/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 23s 116ms/step - accuracy: 0.9811 - loss: 0

In [14]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, TimeDistributed, Bidirectional, Dropout
from tensorflow.keras.models import Model
import tensorflow as tf
from tensorflow.keras.metrics import sparse_categorical_accuracy
from tensorflow.keras.callbacks import EarlyStopping

EMBEDDING_DIM = 128
LSTM_UNITS = 64


# 1. Definir la capa de Entrada
inpusts_1 = Input(shape=(MAX_LEN,), name='word_ids_input')

# 2. Conectar las capas
# La capa Embedding recibe los 'inpusts_1'
x = Embedding(
    input_dim=WORD_VOCAB_SIZE, 
    output_dim=EMBEDDING_DIM, 
    mask_zero=True, 
    name='word_embedding'
)(inpusts_1)



# 3. Primera capa Bidirectional LSTM
x = Bidirectional(
    LSTM(
        units=LSTM_UNITS, 
        return_sequences=True,
        name='bidirectional_lstm_1'
    )
)(x)

# 5. Capa de Salida (igual que antes)
outputs = TimeDistributed(
    Dense(units=TAGS_VOCAB_SIZE, activation='softmax'), 
    name='pos_tag_output'
)(x)

# 6. Crear el Modelo final
model_1 = Model(inputs=inpusts_1, outputs=outputs, name='pos_tagger_model_v2_bidirectional')

# ¡Listo! Ahora puedes imprimir el resumen y ver la nueva arquitectura
model_1.summary()

# Compilar el modelo
model_1.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
# 2. Crea una instancia del callback
early_stopping_callback_acc = EarlyStopping(
    monitor='val_accuracy', 
    patience=3,
    min_delta=0.001,          # Una mejora de 0.1% 
    mode='max',               # Queremos maximizar la precisión (accuracy)
    restore_best_weights=True
)

# 3. Pasa el callback a model.fit()

# Parámetros de entrenamiento
EPOCHS = 50
BATCH_SIZE = 64


# Entrenar el modelo
history_1 = model_1.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_dev, y_dev),
    callbacks=[early_stopping_callback_acc]
)

Model: "pos_tagger_model_v2_bidirectional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ word_ids_input      │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embedding      │ (None, 128, 128)  │  2,131,968 │ word_ids_input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 128)       │          0 │ word_ids_input[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 128, 128)  │     98,816 │ word_embedding[0… │
│ (Bidirectional)     │                   │            │ not_equal_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pos_tag_output      │ (None, 128, 19)   │      2,451 │ bidirectional_3[… │
│ (TimeDistributed)   │                   │            │ not_equal_3[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,233,235 (8.52 MB)

 Trainable params: 2,233,235 (8.52 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 27s 118ms/step - accuracy: 0.6491 - loss: 1.2595 - val_accuracy: 0.8793 - val_loss: 0.4631
Epoch 2/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 25s 125ms/step - accuracy: 0.9326 - loss: 0.2562 - val_accuracy: 0.9119 - val_loss: 0.3133
Epoch 3/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 25s 125ms/step - accuracy: 0.9618 - loss: 0.1398 - val_accuracy: 0.9189 - val_loss: 0.2913
Epoch 4/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 23s 117ms/step - accuracy: 0.9721 - loss: 0.0992 - val_accuracy: 0.9194 - val_loss: 0.2867
Epoch 5/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 24s 120ms/step - accuracy: 0.9783 - loss: 0.0766 - val_accuracy: 0.9204 - val_loss: 0.2937
Epoch 6/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 24s 122ms/step - accuracy: 0.9829 - loss: 0.0612 - val_accuracy: 0.9196 - val_loss: 0.3031
Epoch 7/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 25s 126ms/step - accuracy: 0.9867 - loss: 0.0493 - val_accuracy: 0.9178 - val_loss: 0.3179
Epoch 8/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 24s 123ms/step - accuracy: 0.9895 - loss: 0

In [16]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, TimeDistributed, Bidirectional, Dropout
from tensorflow.keras.models import Model
import tensorflow as tf
from tensorflow.keras.metrics import sparse_categorical_accuracy
from tensorflow.keras.callbacks import EarlyStopping

EMBEDDING_DIM = 128
LSTM_UNITS = 64


# 1. Definir la capa de Entrada
inpusts_1 = Input(shape=(MAX_LEN,), name='word_ids_input')

# 2. Conectar las capas
# La capa Embedding recibe los 'inpusts_1'
x = Embedding(
    input_dim=WORD_VOCAB_SIZE, 
    output_dim=EMBEDDING_DIM, 
    mask_zero=True, 
    name='word_embedding'
)(inpusts_1)



# 3. Primera capa Bidirectional LSTM
x = Bidirectional(
    LSTM(
        units=LSTM_UNITS, 
        return_sequences=True,
        name='bidirectional_lstm_1'
    )
)(x)

# 5. Capa de Salida (igual que antes)
outputs = TimeDistributed(
    Dense(units=TAGS_VOCAB_SIZE, activation='softmax'), 
    name='pos_tag_output'
)(x)

# 6. Crear el Modelo final
model_1 = Model(inputs=inpusts_1, outputs=outputs, name='pos_tagger_model_v2_bidirectional')

# ¡Listo! Ahora puedes imprimir el resumen y ver la nueva arquitectura
model_1.summary()

# Compilar el modelo
model_1.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
# 2. Crea una instancia del callback
early_stopping_callback_acc = EarlyStopping(
    monitor='val_accuracy', 
    patience=3,
    min_delta=0.001,          # Una mejora de 0.1% 
    mode='max',               # Queremos maximizar la precisión (accuracy)
    restore_best_weights=True
)

# 3. Pasa el callback a model.fit()

# Parámetros de entrenamiento
EPOCHS = 50
BATCH_SIZE = 32


# Entrenar el modelo
history_1 = model_1.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_dev, y_dev),
    callbacks=[early_stopping_callback_acc]
)

Model: "pos_tagger_model_v2_bidirectional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ word_ids_input      │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embedding      │ (None, 128, 128)  │  2,131,968 │ word_ids_input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_5         │ (None, 128)       │          0 │ word_ids_input[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_5     │ (None, 128, 128)  │     98,816 │ word_embedding[0… │
│ (Bidirectional)     │                   │            │ not_equal_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pos_tag_output      │ (None, 128, 19)   │      2,451 │ bidirectional_5[… │
│ (TimeDistributed)   │                   │            │ not_equal_5[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,233,235 (8.52 MB)

 Trainable params: 2,233,235 (8.52 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 37s 82ms/step - accuracy: 0.7476 - loss: 0.8822 - val_accuracy: 0.8981 - val_loss: 0.3543
Epoch 2/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 36s 91ms/step - accuracy: 0.9482 - loss: 0.1815 - val_accuracy: 0.9140 - val_loss: 0.2933
Epoch 3/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 35s 90ms/step - accuracy: 0.9687 - loss: 0.1063 - val_accuracy: 0.9183 - val_loss: 0.2812
Epoch 4/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 35s 89ms/step - accuracy: 0.9771 - loss: 0.0768 - val_accuracy: 0.9172 - val_loss: 0.2946
Epoch 5/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 35s 88ms/step - accuracy: 0.9832 - loss: 0.0581 - val_accuracy: 0.9179 - val_loss: 0.3044
Epoch 6/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 46s 117ms/step - accuracy: 0.9876 - loss: 0.0442 - val_accuracy: 0.9179 - val_loss: 0.3249


In [15]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, TimeDistributed, Bidirectional, Dropout
from tensorflow.keras.models import Model
import tensorflow as tf
from tensorflow.keras.metrics import sparse_categorical_accuracy
from tensorflow.keras.callbacks import EarlyStopping

EMBEDDING_DIM = 128
LSTM_UNITS = 128


# 1. Definir la capa de Entrada
inpusts_2 = Input(shape=(MAX_LEN,), name='word_ids_input')

# 2. Conectar las capas
# La capa Embedding recibe los 'inpusts_2'
x = Embedding(
    input_dim=WORD_VOCAB_SIZE, 
    output_dim=EMBEDDING_DIM, 
    mask_zero=True, 
    name='word_embedding'
)(inpusts_2)



# 3. Primera capa Bidirectional LSTM
x = Bidirectional(
    LSTM(
        units=LSTM_UNITS, 
        return_sequences=True,
        name='bidirectional_lstm_1'
    )
)(x)

# 5. Capa de Salida (igual que antes)
outputs = TimeDistributed(
    Dense(units=TAGS_VOCAB_SIZE, activation='softmax'), 
    name='pos_tag_output'
)(x)

# 6. Crear el Modelo final
model_2 = Model(inputs=inpusts_2, outputs=outputs, name='pos_tagger_model_v2_bidirectional')

# ¡Listo! Ahora puedes imprimir el resumen y ver la nueva arquitectura
model_2.summary()

# Compilar el modelo
model_2.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
# 2. Crea una instancia del callback
early_stopping_callback_acc = EarlyStopping(
    monitor='val_accuracy', 
    patience=3,
    min_delta=0.001,          # Una mejora de 0.1% 
    mode='max',               # Queremos maximizar la precisión (accuracy)
    restore_best_weights=True
)

# 3. Pasa el callback a model.fit()

# Parámetros de entrenamiento
EPOCHS = 50
BATCH_SIZE = 64


# Entrenar el modelo
history_2 = model_2.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_dev, y_dev),
    callbacks=[early_stopping_callback_acc]
)

Model: "pos_tagger_model_v2_bidirectional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ word_ids_input      │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embedding      │ (None, 128, 128)  │  2,131,968 │ word_ids_input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, 128)       │          0 │ word_ids_input[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_4     │ (None, 128, 256)  │    263,168 │ word_embedding[0… │
│ (Bidirectional)     │                   │            │ not_equal_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pos_tag_output      │ (None, 128, 19)   │      4,883 │ bidirectional_4[… │
│ (TimeDistributed)   │                   │            │ not_equal_4[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,400,019 (9.16 MB)

 Trainable params: 2,400,019 (9.16 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 41s 189ms/step - accuracy: 0.6602 - loss: 1.1643 - val_accuracy: 0.8847 - val_loss: 0.4111
Epoch 2/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 38s 192ms/step - accuracy: 0.9383 - loss: 0.2237 - val_accuracy: 0.9137 - val_loss: 0.3013
Epoch 3/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 38s 194ms/step - accuracy: 0.9645 - loss: 0.1246 - val_accuracy: 0.9174 - val_loss: 0.2965
Epoch 4/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 35s 178ms/step - accuracy: 0.9742 - loss: 0.0887 - val_accuracy: 0.9190 - val_loss: 0.2941
Epoch 5/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 34s 176ms/step - accuracy: 0.9801 - loss: 0.0674 - val_accuracy: 0.9172 - val_loss: 0.3203
Epoch 6/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 35s 180ms/step - accuracy: 0.9850 - loss: 0.0523 - val_accuracy: 0.9182 - val_loss: 0.3302
Epoch 7/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 35s 180ms/step - accuracy: 0.9888 - loss: 0.0408 - val_accuracy: 0.9170 - val_loss: 0.3381


In [7]:
import numpy as np


# 1. Obtenemos el vocabulario de etiquetas (la lista de strings)
# El ID 0 es '', el ID 1 es '[UNK]', así que el vocabulario real empieza en el índice 2
tags_vocab = tags_vectorizer.get_vocabulary()

def predict_tags(sentence_string):
    """
    Toma una frase (string), la procesa con el modelo 
    y muestra las predicciones palabra por palabra.
    """
    
    print(f"--- Prediciendo para: ---\n{sentence_string}\n")
    
    # 0. Separamos las palabras para saber cuántas son
    words = sentence_string.split(' ')
    
    # 1. Convertir la frase (string) en un tensor de IDs (shape [1, 128])
    # El vectorizador espera una lista de strings
    input_tensor = word_vectorizer([sentence_string])
    print("input_tensor",input_tensor)
    
    # 2. Obtener las predicciones del modelo
    # El modelo devuelve probabilidades (shape [1, 128, 20])
    predictions = model.predict(input_tensor)
    
    # 3. Encontrar el ID de la etiqueta con mayor probabilidad para cada token
    # Usamos np.argmax para obtener los IDs ganadores (shape [1, 128])
    predicted_ids = np.argmax(predictions, axis=-1)[0] # [0] para coger el primer (y único) batch
    print("predicted_ids",predicted_ids)
    
    # 4. Mostrar los resultados
    print("--- Resultados: ---")
    for i in range(len(words)):
        word = words[i]
        tag_id = predicted_ids[i]
        tag_name = tags_vocab[tag_id]
        
        print(f"{word:<15} -> {tag_name}")

In [9]:
# Prueba 1: La frase de ejemplo de la práctica
predict_tags("Google is a nice search engine .")

# Prueba 2: Una frase nueva
predict_tags("I am writing this new sentence for the test .")

# Prueba 3: Otra más
predict_tags("The university is in santiago .")

# Prueba 3: Otra más
predict_tags("telescopio comer")

--- Prediciendo para: ---
Google is a nice search engine .

input_tensor tf.Tensor(
[[2502   11    7  258  891 1894    3    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]], shape=(1, 128), dtype=int64)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
predicted_ids [10  9  7  8  2  2  3 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10
 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10
 10 10 10 10 10 10 10 10 1

# MODELO EN ESPAÑOL